# Aula 1 - LLM APIs

## Template da V1 do projeto

Neste notebook, você vai montar a primeira versão do assistente de estudos da monitoria.

As células com `TODO` são partes que você precisa completar. Não coloque sua API key diretamente no notebook, utilize o `.env`.

## Objetivos

Ao final, você deve conseguir:

- configurar um cliente para a API da Groq;
- montar mensagens com `system` e `user`;
- enviar uma requisição e ler a resposta;
- observar o uso de tokens;
- explicar por que chamadas separadas não têm memória automática;
- testar o efeito da temperatura;
- salvar a V1 em um commit.

## 1. Configuração do ambiente

Antes de executar o notebook:

1. crie um ambiente virtual;
2. instale as dependências de `requirements.txt`;
3. copie `env.example` para `.env`;
4. preencha `GROQ_API_KEY` no `.env`.

In [2]:
# Passo a passo para criar o .venv: 
# Abra o Terminal na raiz do projeto e digite:
# python -m venv venv
# Depois digite: 
# cd "Aula 01"
# pip install -r requirements.txt

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Monte as mensagens

A aplicação controla o comportamento do assistente usando mensagens.

Use uma mensagem `system` para definir o papel do assistente e uma mensagem `user` para fazer uma pergunta sobre a monitoria.

In [4]:
# TODO: crie uma lista chamada `messages` com uma mensagem `system` e uma mensagem `user`.

messages = [
    {
        "role": "system",
        "content": "Voce é o coordenador da monitoria de alunos da faculdade do cleiton, responda as perguntas relacionadas as monitorias de forma curta mas educada." 
    },
    {
        "role": "user",
        "content": "Porque a monitoria é tão cheia?"
    }
]

print(messages)


[{'role': 'system', 'content': 'Voce é o coordenador da monitoria de alunos da faculdade do cleiton, responda as perguntas relacionadas as monitorias de forma curta mas educada.'}, {'role': 'user', 'content': 'Porque a monitoria é tão cheia?'}]


## 3. Faça a primeira chamada

Envie `messages` para o modelo configurado em `MODEL` e salve a resposta em uma variável chamada `response`.

In [6]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
)

print(response)

ChatCompletion(id='chatcmpl-e39225a6-bdca-4820-ad9f-f07476a4eb83', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A monitoria costuma ficar cheia porque há muita demanda dos alunos que buscam reforço e apoio nas disciplinas, enquanto o número de vagas é limitado. Além disso, os horários mais convenientes tendem a atrair ainda mais estudantes. Estamos trabalhando para ampliar as vagas e oferecer mais turmas nos próximos semestres. Obrigado pela compreensão!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks in Portuguese: "Porque a monitoria é tão cheia?" They want a short but polite answer, as coordinator. Should explain reasons: high demand, limited slots, popularity, schedule, etc. Keep short.'))], created=1790818094, model='openai/gpt-oss-120b', object='chat.completion', metadata=None, moderation=None, service_tier='on_demand', system_fingerprint='fp_dee44

## 4. Leia o texto da resposta

A resposta contém uma lista de escolhas. Nesta atividade, vamos usar o texto da primeira escolha.

In [7]:

print(response.choices[0].message.content)

A monitoria costuma ficar cheia porque há muita demanda dos alunos que buscam reforço e apoio nas disciplinas, enquanto o número de vagas é limitado. Além disso, os horários mais convenientes tendem a atrair ainda mais estudantes. Estamos trabalhando para ampliar as vagas e oferecer mais turmas nos próximos semestres. Obrigado pela compreensão!


## 5. Observe o uso de tokens

Além do texto, a API informa quantos tokens entraram na chamada, quantos saíram e qual foi o total.

In [11]:
# TODO: mostre `prompt_tokens`, `completion_tokens` e `total_tokens` usando `response.usage`.
print("Uso de Tokens")
print("Entrada: ", response.usage.prompt_tokens)
print("Saida: ", response.usage.completion_tokens)
print("Total: ", response.usage.total_tokens)

Uso de Tokens
Entrada:  114
Saida:  123
Total:  237


## 6. Teste a ausência de memória

Faça uma chamada dizendo que o nome do seu usuário. Depois faça uma segunda chamada perguntando o nome, mas sem reenviar a primeira mensagem.

Salve a segunda resposta em `sem_historico`. O que aconteceu?

In [ ]:
# TODO: faça duas chamadas independentes e salve a segunda resposta em `sem_historico`.
client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", "content": "Meu nome é Pedro!"}]
)

sem_historico = client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", "content": "Qual o meu nome?"}]
)

In [13]:
print(sem_historico.choices[0].message.content)

Eu não sei o seu nome ainda. 😊 Poderia me dizer como prefere ser chamado?


Agora repita a pergunta, mas envie explicitamente o histórico em `messages`. Inclua uma mensagem `assistant` representando a resposta anterior.

A aplicação cria a sensação de memória ao reenviar o contexto.

In [14]:
# TODO: crie uma nova lista de mensagens com a conversa completa e salve a resposta em `com_historico`.
client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", "content": "Meu nome é Pedro!"}]
)

com_historico = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role": "user", "content": "Meu nome é Pedro!"},
        {"role": "user", "content": "Qual o meu nome?"},
    ],
)

In [15]:
# TODO: mostre o texto de `com_historico` e compare com o resultado sem histórico.
print(sem_historico.choices[0].message.content)
print(com_historico.choices[0].message.content)

Eu não sei o seu nome ainda. 😊 Poderia me dizer como prefere ser chamado?
Seu nome é Pedro.


## 7. Experimente a temperatura

Use o mesmo prompt duas vezes: uma com `temperature=0` e outra com `temperature=1`.

Compare a variedade das respostas. Temperatura baixa tende a ser mais previsível; temperatura alta tende a permitir mais variação.

In [22]:
# TODO: faça as chamadas com duas temperaturas e compare os resultados.
prompt = "Sugira um nome para meu cachorro, seja criativo! || Mande uma lista com apenas 5 nomes e nada mais."

for temperature in [0, 1]:
    response = client.chat.completions.create(
        model = MODEL,
        messages = [{"role": "user", "content": prompt}],
        temperature = temperature
    )
    print(f"temperature = {temperature}: {response.choices[0].message.content}")



temperature = 0: 1. Trovão  
2. Lume  
3. Zéfiro  
4. Pingo  
5. Astro
temperature = 1: 1. Bolt  
2. Luma  
3. Pipo  
4. Zazu  
5. Nix​


## 8. Checkpoint da V1

Antes de fazer o commit, confirme:

- [X] a aplicação chamou a API da Groq;
- [X] a chave ficou no `.env`;
- [X] a requisição usou `system` e `user`;
- [X] a resposta apareceu;
- [X] o uso de tokens foi observado;
- [X] você comparou chamadas sem histórico e com histórico;
- [X] você testou duas temperaturas.
